In [1]:
import pandas as pd

# Load datasets
enron = pd.read_csv("../data/raw/enron.csv")
nazario = pd.read_csv("../data/raw/nazario.csv")

# Create unified text column
enron["text"] = enron["subject"].fillna("") + " " + enron["body"].fillna("")
nazario["text"] = nazario["subject"].fillna("") + " " + nazario["body"].fillna("")

# Keep only necessary columns
enron = enron[["text", "label"]]
nazario = nazario[["text", "label"]]

# Add source labels
enron["source"] = "enron"
nazario["source"] = "nazario"

# Combine datasets
merged = pd.concat([enron, nazario], ignore_index=True)

print("Merged shape:", merged.shape)
merged.head()



Merged shape: (31332, 3)


,text,label,source
0,"hpl nom for may 25 , 2001 ( see attached file ...",0,enron
1,re : nom / actual vols for 24 th - - - - - - -...,0,enron
2,"enron actuals for march 30 - april 1 , 201 est...",0,enron
3,"hpl nom for may 30 , 2001 ( see attached file ...",0,enron
4,"hpl nom for june 1 , 2001 ( see attached file ...",0,enron


In [2]:
merged.to_csv("../data/processed/clean_emails.csv", index=False)
print("Saved successfully")


Saved successfully


In [3]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", " ", text)   # remove urls
    text = re.sub(r"[^a-z\s]", " ", text)  # remove punctuation & numbers
    text = re.sub(r"\s+", " ", text)       # remove extra spaces
    return text.strip()

merged["clean_text"] = merged["text"].apply(clean_text)

merged[["text", "clean_text"]].head()


,text,clean_text
0,"hpl nom for may 25 , 2001 ( see attached file ...",hpl nom for may see attached file hplno xls hp...
1,re : nom / actual vols for 24 th - - - - - - -...,re nom actual vols for th forwarded by sabrae ...
2,"enron actuals for march 30 - april 1 , 201 est...",enron actuals for march april estimated actual...
3,"hpl nom for may 30 , 2001 ( see attached file ...",hpl nom for may see attached file hplno xls hp...
4,"hpl nom for june 1 , 2001 ( see attached file ...",hpl nom for june see attached file hplno xls h...


In [4]:
# Remove duplicate emails
before = len(merged)
merged = merged.drop_duplicates(subset="clean_text")
after = len(merged)

print("Removed duplicates:", before - after)
print("Final dataset size:", after)


Removed duplicates: 1499
Final dataset size: 29833


In [5]:
# Save processed dataset
merged.to_csv("../data/processed/clean_emails.csv", index=False)
print("Saved to data/processed/clean_emails.csv")


Saved to data/processed/clean_emails.csv


In [6]:
merged["label"].value_counts()


label
1    15447
0    14386
Name: count, dtype: int64

In [7]:
# Check label distribution in each dataset

print("ENRON:")
print(enron['label'].value_counts())

print("\nNAZARIO:")
print(nazario['label'].value_counts())

print("\nMERGED:")
print(merged['label'].value_counts())


ENRON:
label
0    15791
1    13976
Name: count, dtype: int64

NAZARIO:
label
1    1565
Name: count, dtype: int64

MERGED:
label
1    15447
0    14386
Name: count, dtype: int64
